In [10]:

import os
print(os.listdir())

['.config', 'best_model .pkl', 'cleaned_data.csv', 'sample_data']


In [18]:

import os
import re
import json
import joblib
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# =====================================================================
# STEP 1: SET UP THE LLM API CONNECTION
# =====================================================================

API_KEY = os.environ.get('API_KEY',"ENTER YOUR API_KEY")

API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_NAME = "openai/gpt-4o-mini"


def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code != 200:
        print(f"LLM call failed. Status code: {response.status_code}")
        print(response.text)
        return None

    data = response.json()
    return data["choices"][0]["message"]["content"]

# -------------------------------------------------------------------
# Demonstrate call_llm() with a simple test prompt
# -------------------------------------------------------------------
print("=" * 70)
print("STEP 1 DEMO: Testing call_llm() with a simple prompt")
print("=" * 70)
test_response = call_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="Reply with only the word: hello",
    temperature=0.0
)
print("Response:", test_response)


# =====================================================================
# STEP 2: PROMPT DESIGN
# =====================================================================


SYSTEM_PROMPT = """You are a clinical risk-model explainer. You will be given a patient's \
feature values, a machine learning model's predicted class for heart disease risk \
(Yes or No), and the model's predicted probability for that class. Your job is to \
produce a short, plain-language explanation grounded strictly in the feature values \
provided. Do not invent facts that are not present in the input, and do not provide \
medical diagnosis or treatment instructions -- only describe what the model's inputs \
suggest.

Respond with ONLY a single valid JSON object. Do not include markdown formatting, \
code fences, or any text before or after the JSON object.

The JSON object must have exactly these five fields:
- "prediction_label": string, a short label such as "Elevated risk of heart disease" \
or "Lower risk of heart disease"
- "confidence_level": string, one of "low", "medium", or "high", based on how far the \
probability is from 0.5
- "top_reason": string, the single feature value most likely driving this prediction
- "second_reason": string, the second most likely contributing feature value
- "next_step": string, a brief, sensible next action (e.g. recommending a follow-up \
screening), not a diagnosis or treatment plan
"""


# USER PROMPT TEMPLATE (placeholders shown as {feature_values},{predicted_class}, {predicted_probability})

USER_PROMPT_TEMPLATE = """Patient feature values: {feature_values}
Model predicted class: {predicted_class}
Model predicted probability of the positive class: {predicted_probability}

Provide your JSON explanation now."""


def build_user_prompt(patient_features, predicted_class, predicted_probability):
    return USER_PROMPT_TEMPLATE.format(
        feature_values=json.dumps(patient_features),
        predicted_class=predicted_class,
        predicted_probability=predicted_probability
    )



# =====================================================================
# STEP 3: Guardrails
# =====================================================================


def has_pii(text):

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))


def safe_call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """Wraps call_llm() with the PII guardrail check."""
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None
    return call_llm(system_prompt, user_prompt, temperature, max_tokens)


print("\n" + "=" * 70)
print("STEP 3 DEMO: PII guardrail test")
print("=" * 70)

pii_test_input = ("Patient contact info: jane.smith@example.com. "
                   "Age=71, BMI=33.2, SystolicBP=162.")
clean_test_input = "Age=71, BMI=33.2, SystolicBP=162, DiastolicBP=98."

print("\n--- Test 1: input WITH an email address (should be BLOCKED) ---")
result_1 = safe_call_llm("You are a helpful assistant.", pii_test_input)
print("Result:", result_1)

print("\n--- Test 2: input WITHOUT PII (should PROCEED to the LLM) ---")
result_2 = safe_call_llm("You are a helpful assistant.", clean_test_input)
print("Result:", result_2)


# =====================================================================
# STEP 4: LOAD THE MODEL AND DEFINE encode_record()
# =====================================================================

print("\n" + "=" * 70)
print("STEP 4: LOAD best_model.pkl AND ENCODE FEATURE INPUTS")
print("=" * 70)

model = joblib.load('best_model.pkl')
print("Model loaded successfully:", type(model).__name__)


FEATURE_COLUMNS = [
    'Age', 'BMI', 'SystolicBP', 'DiastolicBP', 'GlucoseMgDl', 'RestingHR',
    'ExerciseHrsPerWeek', 'Sex_Male', 'Smoker_Yes', 'BloodType_A-',
    'BloodType_AB+', 'BloodType_AB-', 'BloodType_B+', 'BloodType_B-',
    'BloodType_O+', 'BloodType_O-', 'FamilyHistory_Yes', 'Diabetes_Yes'
]


def encode_record(features: dict) -> pd.DataFrame:

    row = {
        'Age': features['Age'],
        'BMI': features['BMI'],
        'SystolicBP': features['SystolicBP'],
        'DiastolicBP': features['DiastolicBP'],
        'GlucoseMgDl': features['GlucoseMgDl'],
        'RestingHR': features['RestingHR'],
        'ExerciseHrsPerWeek': features['ExerciseHrsPerWeek'],
        'Sex_Male': features['Sex'] == 'Male',
        'Smoker_Yes': features['Smoker'] == 'Yes',
        'FamilyHistory_Yes': features['FamilyHistory'] == 'Yes',
        'Diabetes_Yes': features['Diabetes'] == 'Yes',
    }
    # BloodType_A+ was dropped during training (drop_first=True), so it's
    # the "reference" category -- a patient with BloodType A+ simply gets
    # False for every BloodType_ column below.
    for blood_type in ['A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']:
        row[f'BloodType_{blood_type}'] = (features['BloodType'] == blood_type)

    return pd.DataFrame([row])[FEATURE_COLUMNS]


# Three hand-crafted patients to run through the whole pipeline
test_patients = [
    {  # Patient 1: several risk factors present
        'Age': 71, 'BMI': 33.2, 'SystolicBP': 162, 'DiastolicBP': 98,
        'GlucoseMgDl': 155, 'RestingHR': 88, 'ExerciseHrsPerWeek': 0.2,
        'Sex': 'Male', 'Smoker': 'Yes', 'BloodType': 'O+',
        'FamilyHistory': 'Yes', 'Diabetes': 'Yes'
    },
    {  # Patient 2: healthy profile, low risk
        'Age': 26, 'BMI': 21.0, 'SystolicBP': 108, 'DiastolicBP': 68,
        'GlucoseMgDl': 85, 'RestingHR': 60, 'ExerciseHrsPerWeek': 6.0,
        'Sex': 'Female', 'Smoker': 'No', 'BloodType': 'A+',
        'FamilyHistory': 'No', 'Diabetes': 'No'
    },
    {  # Patient 3: a more borderline / mixed profile
        'Age': 52, 'BMI': 27.4, 'SystolicBP': 134, 'DiastolicBP': 84,
        'GlucoseMgDl': 110, 'RestingHR': 74, 'ExerciseHrsPerWeek': 2.0,
        'Sex': 'Male', 'Smoker': 'No', 'BloodType': 'B+',
        'FamilyHistory': 'Yes', 'Diabetes': 'No'
    }
]

model_outputs = []
for i, patient in enumerate(test_patients):
    encoded_row = encode_record(patient)
    predicted_class = int(model.predict(encoded_row)[0])
    predicted_proba = float(model.predict_proba(encoded_row)[0, 1])
    model_outputs.append({
        'patient': patient,
        'predicted_class': 'Yes' if predicted_class == 1 else 'No',
        'predicted_probability': round(predicted_proba, 3)
    })
    print(f"\nPatient {i+1}: predicted class = "
          f"{'Yes' if predicted_class == 1 else 'No'}, "
          f"probability = {predicted_proba:.3f}")


# =====================================================================
# STEP 5: JSON SCHEMA FOR STRUCTURED OUTPUT VALIDATION
# =====================================================================

EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["prediction_label", "confidence_level", "top_reason",
                 "second_reason", "next_step"]
}

FALLBACK_EXPLANATION = {
    "prediction_label": None,
    "confidence_level": None,
    "top_reason": None,
    "second_reason": None,
    "next_step": None
}


def parse_and_validate(raw_response):

    if raw_response is None:
        return dict(FALLBACK_EXPLANATION), "fail (no response / blocked)"

    try:
        parsed = json.loads(raw_response.strip())
    except json.JSONDecodeError as e:
        print(f"  JSON parsing failed: {e}")
        return dict(FALLBACK_EXPLANATION), f"fail (JSONDecodeError: {e})"

    if JSONSCHEMA_AVAILABLE:
        try:
            jsonschema.validate(instance=parsed, schema=EXPLANATION_SCHEMA)
        except ValidationError as e:
            print(f"  Schema validation failed: {e.message}")
            return dict(FALLBACK_EXPLANATION), f"fail (ValidationError: {e.message})"

    return parsed, "pass"


# =====================================================================
# STEP 6: RUN THE FULL PIPELINE ON ALL 3 PATIENTS (temperature=0)
# =====================================================================

print("\n" + "=" * 70)
print("STEP 6: FULL PIPELINE - PREDICTION + LLM EXPLANATION (temperature=0)")
print("=" * 70)

demonstration_rows = []
for i, output in enumerate(model_outputs):
    patient = output['patient']
    predicted_class = output['predicted_class']
    predicted_probability = output['predicted_probability']

    user_prompt = build_user_prompt(patient, predicted_class, predicted_probability)

    print(f"\n--- Patient {i+1} ---")
    print("Feature input:", patient)
    print("Predicted class:", predicted_class, " Probability:", predicted_probability)

    # PII guardrail check before every call
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        raw_response = None
    else:
        raw_response = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0)

    print("Raw LLM response:", raw_response)

    parsed_explanation, status = parse_and_validate(raw_response)
    print("Validation status:", status)
    print("Parsed explanation:", parsed_explanation)

    demonstration_rows.append({
        'Patient': f'Patient {i+1}',
        'Feature Input': patient,
        'Predicted Class': predicted_class,
        'Probability': predicted_probability,
        'LLM Output (raw)': raw_response,
        'Explanation JSON': parsed_explanation,
        'Valid JSON': 'pass' if status == 'pass' else 'fail',
        'Validation Status': status
    })


# =====================================================================
# STEP 7: TEMPERATURE A/B COMPARISON (temp=0 vs temp=0.7)
# =====================================================================

print("\n" + "=" * 70)
print("STEP 7: TEMPERATURE A/B COMPARISON (temp=0 vs temp=0.7)")
print("=" * 70)

temperature_comparison_rows = []
for i, output in enumerate(model_outputs):
    patient = output['patient']
    predicted_class = output['predicted_class']
    predicted_probability = output['predicted_probability']
    user_prompt = build_user_prompt(patient, predicted_class, predicted_probability)

    # Re-use the temp=0 response already generated in Step 6 for consistency
    response_temp0 = demonstration_rows[i]['LLM Output (raw)']

    response_temp07 = None
    if not has_pii(user_prompt):
        response_temp07 = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.7)

    print(f"\n--- Patient {i+1} ---")
    print("temp=0.0 response:", response_temp0)
    print("temp=0.7 response:", response_temp07)

    temperature_comparison_rows.append({
        'Patient': f'Patient {i+1}',
        'Output (temp=0)': response_temp0,
        'Output (temp=0.7)': response_temp07
    })

print("\n--- Why temperature changes the output ---")
print("At temperature=0, the model always selects its single highest-probability "
      "next word at each step, which is why the same input reliably produces the "
      "same (or nearly the same) output every time. At temperature=0.7, the model "
      "instead samples from a wider range of plausible next words, weighted by "
      "their probabilities, which introduces variety -- the model might phrase the "
      "explanation differently, choose a different 'top_reason', or vary its wording "
      "each time you re-run the exact same prompt.")


# =====================================================================
# FINAL SUMMARY TABLES (for quick reference / copying into the README)
# =====================================================================

print("\n" + "=" * 70)
print("SUMMARY: 3-ROW DEMONSTRATION TABLE")
print("=" * 70)
summary_df = pd.DataFrame([{
    'Input': f"Patient {i+1}",
    'LLM Output': row['LLM Output (raw)'],
    'Valid JSON': row['Valid JSON'],
    'Pass/Block': 'pass' if row['LLM Output (raw)'] is not None else 'blocked'
} for i, row in enumerate(demonstration_rows)])
print(summary_df.to_string(index=False))



STEP 1 DEMO: Testing call_llm() with a simple prompt
Response: hello

STEP 3 DEMO: PII guardrail test

--- Test 1: input WITH an email address (should be BLOCKED) ---
Input blocked: PII detected.
Result: None

--- Test 2: input WITHOUT PII (should PROCEED to the LLM) ---
Result: Based on the information provided, here are some insights regarding the individual's health metrics:

1. **Age**: 71 years old.
2. **BMI (Body Mass Index)**: 33.2
   - A BMI of 33.2 falls into the category of obesity (specifically, Class 1 obesity, which is defined as a BMI between 30 and 34.9). This can increase the risk of various health issues, including heart disease, diabetes, and certain cancers.
3. **Blood Pressure**: 
   - Systolic BP: 162 mmHg
   - Diastolic BP: 98 mmHg
   - This blood pressure reading is classified as Stage 2 Hypertension (systolic ≥ 140 mmHg or diastolic ≥ 90 mmHg). Stage 2 hypertension is a serious condition that requires medical attention and management.

### Recommendations:
- **C